In [ ]:
!pip install folium requests groq ipywidgets -q
print("✅ Hazır!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 65.1 MB/s eta 0:00:00
✅ Hazır!


In [ ]:
import folium
import requests
import json
import groq as groq_lib
from datetime import date
from IPython.display import display, HTML

In [1]:
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"
print("✅ API Key ayarlandı!")

✅ API Key ayarlandı!


In [ ]:
def get_climate_data(latitude, longitude):
    end_year = date.today().year - 1
    params = {
        "latitude": latitude, "longitude": longitude,
        "start_date": f"{end_year}-01-01", "end_date": f"{end_year}-12-31",
        "daily": ["temperature_2m_mean", "precipitation_sum"],
        "timezone": "auto",
    }
    try:
        resp = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=15)
        data = resp.json()["daily"]
        temps = [t for t in data.get("temperature_2m_mean", []) if t]
        precip = [p for p in data.get("precipitation_sum", []) if p]
        return {
            "avg_temp": round(sum(temps)/len(temps), 1) if temps else 15.0,
            "annual_precip": round(sum(precip), 1) if precip else 500.0
        }
    except:
        return {"avg_temp": 15.0, "annual_precip": 500.0}

def get_soil_data(latitude, longitude):
    params = {"lon": longitude, "lat": latitude, "property": ["phh2o"], "depth": ["0-5cm", "5-15cm"], "value": "mean"}
    try:
        resp = requests.get("https://rest.isric.org/soilgrids/v2.0/properties/query", params=params, timeout=20)
        ph_vals = []
        for layer in resp.json().get("properties", {}).get("layers", []):
            for d in layer.get("depths", []):
                v = d.get("values", {}).get("mean")
                if v: ph_vals.append(v/10)
        return {"soil_ph": round(sum(ph_vals)/len(ph_vals), 1) if ph_vals else 6.5}
    except:
        return {"soil_ph": 6.5}

def calculate_suitability(crop, avg_temp, annual_precip, soil_ph, requirements):
    def score(val, mn, mx, amn, amx):
        if mn <= val <= mx:
            # Optimal aralıkta bile tam ortaya yakın olmak gerekiyor
            mid = (mn + mx) / 2
            distance = abs(val - mid) / ((mx - mn) / 2)
            return round(100 - distance * 15)  # max 100, optimalde bile 85-100 arası
        if val < mn: return max(0, round(100*(val-amn)/(mn-amn))) if amn < mn else 0
        return max(0, round(100*(amx-val)/(amx-mx))) if amx > mx else 0
    r = requirements
    t  = score(avg_temp,      r["temp_min"],   r["temp_max"],   r["temp_min"]-8,   r["temp_max"]+8)
    p  = score(annual_precip, r["precip_min"],  r["precip_max"], r["precip_min"]-150, r["precip_max"]+300)
    ph = score(soil_ph,       r["ph_min"],      r["ph_max"],     r["ph_min"]-1.5,   r["ph_max"]+1.5)
    composite = round(0.4*t + 0.35*p + 0.25*ph)
    rating = "Excellent" if composite>=75 else "Good" if composite>=55 else "Moderate" if composite>=35 else "Poor"
    return {"suitability_score": composite, "rating": rating,
            "factors": {"Temperature": round(t), "Precipitation": round(p), "Soil pH": round(ph)}}


print("✅ Araçlar hazır!")

✅ Araçlar hazır!


In [ ]:
CROP_REQUIREMENTS = {
    "wheat":     {"temp_min": 5,  "temp_max": 24, "precip_min": 300, "precip_max": 900,  "ph_min": 6.0, "ph_max": 7.5},
    "corn":      {"temp_min": 15, "temp_max": 32, "precip_min": 500, "precip_max": 1200, "ph_min": 5.8, "ph_max": 7.0},
    "sunflower": {"temp_min": 16, "temp_max": 30, "precip_min": 400, "precip_max": 900,  "ph_min": 6.0, "ph_max": 7.5},
    "tomato":    {"temp_min": 18, "temp_max": 29, "precip_min": 400, "precip_max": 800,  "ph_min": 6.0, "ph_max": 7.0},
    "potato":    {"temp_min": 10, "temp_max": 22, "precip_min": 500, "precip_max": 1000, "ph_min": 5.0, "ph_max": 6.5},
    "barley":    {"temp_min": 5,  "temp_max": 22, "precip_min": 250, "precip_max": 700,  "ph_min": 6.0, "ph_max": 8.0},
    "soybean":   {"temp_min": 20, "temp_max": 30, "precip_min": 450, "precip_max": 900,  "ph_min": 6.0, "ph_max": 7.0},
    "grape":     {"temp_min": 12, "temp_max": 28, "precip_min": 300, "precip_max": 700,  "ph_min": 5.5, "ph_max": 7.0},
}

def run_agent(latitude, longitude, crop):
    print(f"🤖 Agent başlatıldı: {crop} @ ({latitude:.4f}, {longitude:.4f})")

    print("📡 Tool 1: İklim verisi çekiliyor...")
    climate = get_climate_data(latitude, longitude)
    print(f"   ✅ Sıcaklık: {climate['avg_temp']}°C | Yağış: {climate['annual_precip']}mm")

    print("🌱 Tool 2: Toprak verisi çekiliyor...")
    soil = get_soil_data(latitude, longitude)
    print(f"   ✅ Toprak pH: {soil['soil_ph']}")

    print("📊 Tool 3: Uygunluk skoru hesaplanıyor...")
    score = calculate_suitability(crop, climate["avg_temp"], climate["annual_precip"],
                                   soil["soil_ph"], CROP_REQUIREMENTS[crop])
    print(f"   ✅ Skor: {score['suitability_score']}/100 — {score['rating']}")

    print("💬 Groq AI Agent öneri yazıyor...")
    client = groq_lib.Groq(api_key=GROQ_API_KEY)
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a GeoAI agricultural expert agent."},
            {"role": "user", "content": f"""Analyze and give a 3-4 sentence practical recommendation.

Crop: {crop} | Location: {latitude:.4f}, {longitude:.4f}
Temperature: {climate['avg_temp']}°C | Precipitation: {climate['annual_precip']}mm | Soil pH: {soil['soil_ph']}
Suitability Score: {score['suitability_score']}/100 | Rating: {score['rating']}
Factors: {score['factors']}"""}
        ]
    )
    recommendation = response.choices[0].message.content
    print("   ✅ Öneri hazır!")

    return {
        "crop": crop, "latitude": latitude, "longitude": longitude,
        "avg_temp": climate["avg_temp"], "annual_precip": climate["annual_precip"],
        "soil_ph": soil["soil_ph"], "suitability_score": score["suitability_score"],
        "rating": score["rating"], "factors": score["factors"],
        "recommendation": recommendation
    }

print("✅ Groq Agent hazır!")

✅ Groq Agent hazır!


In [ ]:
def show_results(result):
    score = result["suitability_score"]
    color = "#2e7d32" if score>=75 else "#f57f17" if score>=55 else "#e65100" if score>=35 else "#b71c1c"
    emoji = "✅" if score>=75 else "👍" if score>=55 else "⚠️" if score>=35 else "❌"

    html = f"""
    <div style="font-family:Arial; max-width:800px; margin:auto;">
      <div style="background:linear-gradient(135deg,#1b5e20,#2e7d32); color:white; padding:20px; border-radius:12px; margin-bottom:16px;">
        <h2 style="margin:0;">🌾 AI Crop Suitability Agent</h2>
        <p style="margin:4px 0; opacity:0.8;">{result['crop'].upper()} @ ({result['latitude']:.4f}, {result['longitude']:.4f})</p>
      </div>
      <div style="display:flex; gap:12px; margin-bottom:16px;">
        <div style="background:{color}; color:white; padding:20px; border-radius:12px; text-align:center; flex:1;">
          <div style="font-size:52px; font-weight:bold;">{score}</div>
          <div style="font-size:12px; opacity:0.9;">SUITABILITY SCORE / 100</div>
          <div style="font-size:18px; margin-top:8px;">{emoji} {result['rating']}</div>
        </div>
        <div style="background:#f5f5f5; padding:20px; border-radius:12px; flex:2;">
          <table style="width:100%; border-collapse:collapse;">
            <tr><td style="padding:8px; color:#555;">🌡️ Avg Temperature</td><td style="font-weight:bold;">{result['avg_temp']} °C</td></tr>
            <tr><td style="padding:8px; color:#555;">🌧️ Annual Precipitation</td><td style="font-weight:bold;">{result['annual_precip']} mm</td></tr>
            <tr><td style="padding:8px; color:#555;">🌱 Soil pH</td><td style="font-weight:bold;">{result['soil_ph']}</td></tr>
          </table>
        </div>
      </div>
      <div style="background:#f5f5f5; padding:16px; border-radius:12px; margin-bottom:16px;">
        <h4 style="margin:0 0 12px; color:#333;">📊 Factor Scores</h4>
        {"".join([f'<div style="margin:8px 0;"><span style="display:inline-block;width:140px;color:#555;">{"🌡️" if k=="Temperature" else "🌧️" if k=="Precipitation" else "🌱"} {k}</span><div style="display:inline-block;width:300px;background:#ddd;border-radius:4px;height:20px;vertical-align:middle;"><div style="width:{v}%;background:{"#2e7d32" if v>=70 else "#f57f17" if v>=40 else "#b71c1c"};height:100%;border-radius:4px;"></div></div><span style="margin-left:8px;font-weight:bold;">{v}/100</span></div>' for k,v in result['factors'].items()])}
      </div>
      <div style="background:#e8f5e9; border-left:4px solid #2e7d32; padding:16px; border-radius:0 12px 12px 0;">
        <h4 style="margin:0 0 8px; color:#1b5e20;">🤖 AI Agent Recommendation</h4>
        <p style="margin:0; color:#333; line-height:1.6;">{result['recommendation']}</p>
      </div>
    </div>
    """
    display(HTML(html))

    icon_color = "green" if score>=75 else "orange" if score>=40 else "red"
    m = folium.Map(location=[result["latitude"], result["longitude"]], zoom_start=9, tiles="CartoDB positron")
    folium.Marker(
        [result["latitude"], result["longitude"]],
        popup=f"{result['crop']}: {score}/100",
        tooltip=f"Score: {score}/100 — {result['rating']}",
        icon=folium.Icon(color=icon_color, icon="leaf", prefix="fa")
    ).add_to(m)
    folium.Circle([result["latitude"], result["longitude"]], radius=25000,
                  color=icon_color, fill=True, fill_opacity=0.15).add_to(m)
    display(m)

print("✅ Görselleştirme hazır!")

✅ Görselleştirme hazır!


In [ ]:
LATITUDE  = 38.4192   # Konya
LONGITUDE = 32.5300
CROP      = "wheat"   # wheat, corn, sunflower, tomato, potato, barley, soybean, grape

result = run_agent(LATITUDE, LONGITUDE, CROP)
show_results(result)

🤖 Agent başlatıldı: wheat @ (38.4192, 32.5300)
📡 Tool 1: İklim verisi çekiliyor...
   ✅ Sıcaklık: 13.3°C | Yağış: 249.4mm
🌱 Tool 2: Toprak verisi çekiliyor...
   ✅ Toprak pH: 7.8
📊 Tool 3: Uygunluk skoru hesaplanıyor...
   ✅ Skor: 82/100 — Excellent
💬 Groq AI Agent öneri yazıyor...
   ✅ Öneri hazır!


🌡️ Avg Temperature,13.3 °C
🌧️ Annual Precipitation,249.4 mm
🌱 Soil pH,7.8


In [ ]:
LATITUDE  = 38.4189
LONGITUDE = 27.1287
CROP      = "tomato"

result = run_agent(LATITUDE, LONGITUDE, CROP)
show_results(result)

🤖 Agent başlatıldı: tomato @ (38.4189, 27.1287)
📡 Tool 1: İklim verisi çekiliyor...
   ✅ Sıcaklık: 19.4°C | Yağış: 523.6mm
🌱 Tool 2: Toprak verisi çekiliyor...
   ✅ Toprak pH: 6.5
📊 Tool 3: Uygunluk skoru hesaplanıyor...
   ✅ Skor: 94/100 — Excellent
💬 Groq AI Agent öneri yazıyor...
   ✅ Öneri hazır!


🌡️ Avg Temperature,19.4 °C
🌧️ Annual Precipitation,523.6 mm
🌱 Soil pH,6.5


In [ ]:
LATITUDE  = 41.0015
LONGITUDE = 39.7178
CROP      = "corn"

result = run_agent(LATITUDE, LONGITUDE, CROP)
show_results(result)

🤖 Agent başlatıldı: corn @ (41.0015, 39.7178)
📡 Tool 1: İklim verisi çekiliyor...
   ✅ Sıcaklık: 15.1°C | Yağış: 1121.0mm
🌱 Tool 2: Toprak verisi çekiliyor...
   ✅ Toprak pH: 6.2
📊 Tool 3: Uygunluk skoru hesaplanıyor...
   ✅ Skor: 89/100 — Excellent
💬 Groq AI Agent öneri yazıyor...
   ✅ Öneri hazır!


🌡️ Avg Temperature,15.1 °C
🌧️ Annual Precipitation,1121.0 mm
🌱 Soil pH,6.2
